# rabi discovery probability
See the repository README for the paper configurations and cached-data reproduction commands.


In [ ]:
from __future__ import annotations

# Matplotlib backend is selected by the caller.

from dataclasses import fields
from pathlib import Path
import json
import math
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import brentq
from scipy.stats import binom, chi2, ncx2, norm, poisson


def find_repo_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "src" / "dm_search").exists():
            return path
    raise FileNotFoundError("Could not find repo root containing src/dm_search")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
SRC_PATH = REPO_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from dm_search.rabi import (  # noqa: E402
    RabiExperimentConfig,
    RabiParticleSpec,
    RabiReadoutConfig,
    binomial_neyman_pearson_power,
    longitudinal_relaxation_factor,
    rabi_probability_one,
    transverse_relaxation_factor,
)
from dm_search.statistics import null_threshold  # noqa: E402
from dm_search.rabi_prefix_cache import (  # noqa: E402
    assert_matching_raw_setup,
    build_rabi_prefix_setup,
    rabi_prefix_cache_path,
    stable_setup_hash,
)

plt.style.use("seaborn-v0_8-whitegrid")

# Shared manuscript typography and frameless legends.
plt.style.use(REPO_ROOT / "plot_final" / "rabi_paper.mplstyle")


In [ ]:
# Preserve an optional shell override before clearing stale notebook state.
_requested_discovery_preset = os.environ.get("RABI_DISCOVERY_PRESET")
_requested_detector_profile = (
    os.environ.get("RABI_DISCOVERY_DETECTOR_PROFILE")
    or os.environ.get("RABI_PREFIX_DETECTOR_PROFILE")
)

# Clear stale Rabi settings left in this Jupyter kernel by an earlier run.
for key in tuple(os.environ):
    if key.startswith("RABI_PREFIX_") or key.startswith("RABI_DISCOVERY_"):
        os.environ.pop(key)

# ------------------------------------------------------------------
# Raw-data setup: use the same values in rabi_prefix_raw_data.ipynb.
# ------------------------------------------------------------------
# Set False in both prefix_raw_data and discovery_probability for production.
SMALL_CHECK = False
RAW_PRESET = "final" if SMALL_CHECK else (_requested_discovery_preset or "final")
os.environ["RABI_PREFIX_PRESET"] = RAW_PRESET
os.environ["RABI_DISCOVERY_PRESET"] = RAW_PRESET

# Detector profile: current=(C=0.99, epsilon=0.005), high_fidelity=(0.999, 0.0005).
DETECTOR_PROFILE = ("current" if SMALL_CHECK else (_requested_detector_profile or "current")).strip().lower().replace("-", "_")
DETECTOR_PROFILE = {"higher_fidelity": "high_fidelity"}.get(DETECTOR_PROFILE, DETECTOR_PROFILE)
os.environ["RABI_PREFIX_DETECTOR_PROFILE"] = DETECTOR_PROFILE
os.environ["RABI_PREFIX_SENSING_TIME"] = "1.0"
os.environ["RABI_PREFIX_T1"] = "1.0"
os.environ["RABI_PREFIX_T2"] = "1.0"
os.environ["RABI_PREFIX_ADAPTIVE_PERIODS"] = "1,2"  # current v4 request; high_fidelity may use archived L=8
os.environ["RABI_PREFIX_INCLUDE_STEIN"] = "0"

# Uncomment only when overriding the selected preset.
# os.environ["RABI_PREFIX_N_SHOT_MAX"] = "16384"
# os.environ["RABI_PREFIX_N_EXPERIMENTS"] = "1000"
# Current/final default; uncomment only to override it explicitly.
# os.environ["RABI_PREFIX_PHI_R_LIST"] = "0.05,0.08,0.10,0.12,0.15,0.19,0.24,0.32,0.40,0.50,0.60"
# os.environ["RABI_PREFIX_AMPLITUDE_COUNT"] = "48"
# os.environ["RABI_PREFIX_DRIVE_PHASE_COUNT"] = "96"
# os.environ["RABI_PREFIX_ADAPTIVE_PERIODS"] = "1,2"
# os.environ["RABI_PREFIX_CONTRAST"] = "0.99"  # explicit profile override
# os.environ["RABI_PREFIX_READOUT_ERROR"] = "0.005"

PREFIX_ENVIRON = dict(os.environ)
if SMALL_CHECK:
    PREFIX_ENVIRON.update({
        "RABI_PREFIX_DETECTOR_PROFILE": DETECTOR_PROFILE,
        "RABI_PREFIX_N_EXPERIMENTS": "128",
        "RABI_PREFIX_N_NULL": "256",
        "RABI_PREFIX_BATCH_SIZE": "64",
        "RABI_PREFIX_AMPLITUDE_COUNT": "12",
        "RABI_PREFIX_DRIVE_PHASE_COUNT": "24",
        "RABI_PREFIX_CANDIDATE_ANGLE_COUNT": "8",
        "RABI_PREFIX_CANDIDATE_POLAR_COUNT": "5",
        "RABI_PREFIX_ADAPTIVE_PERIODS": "1",
    })

# ------------------------------------------------------------------
# Plot-only setup: changing these values reuses the same raw cache.
# ------------------------------------------------------------------
os.environ["RABI_DISCOVERY_Z_STARS"] = "1.645"
os.environ["RABI_DISCOVERY_TARGET_POWERS"] = "0.5,0.7,0.9,0.95"
os.environ["RABI_DISCOVERY_BOOTSTRAP_DRAWS"] = "4000"
os.environ["RABI_DISCOVERY_POWER_MAX_CHECKPOINTS"] = "8"
os.environ["RABI_DISCOVERY_FPOL_PHASES"] = (
    "0.08,0.14,0.24" if RAW_PRESET == "sample"
    else ("0.06,0.10,0.16,0.25" if RAW_PRESET == "quick" else "0.05,0.08,0.12,0.19")
)

# Optional explicit overrides. Normally the setup hash selects both paths.
# os.environ["RABI_DISCOVERY_RAW_CACHE"] = "plot_final/cache/....npz"
# os.environ["RABI_DISCOVERY_OUTPUT_TAG"] = "my_comparison_label"


In [ ]:
def parse_float_list(text: str) -> np.ndarray:
    return np.asarray([float(piece.strip()) for piece in text.split(",") if piece.strip()], dtype=float)


def parse_int_list(text: str) -> np.ndarray:
    return np.asarray([int(piece.strip()) for piece in text.split(",") if piece.strip()], dtype=int)


RUN_PRESET = os.environ.get("RABI_DISCOVERY_PRESET", "final")
OUTPUT_BASENAME = "rabi_discovery_probability"
RAW_PREFIX_BASENAME = "rabi_prefix_sensitivity_vs_shots"
CACHE_DIR = REPO_ROOT / "plot_final" / "cache"
FIGURE_DIR = REPO_ROOT / "plot_final" / "figure"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# The current dataset is identified by the complete canonical RABI_PREFIX_* setup.
REQUESTED_PREFIX_SETUP = build_rabi_prefix_setup(RUN_PRESET, environ=PREFIX_ENVIRON)
REQUESTED_RAW_SETUP = REQUESTED_PREFIX_SETUP.raw_setup
REQUESTED_RAW_SETUP_HASH = stable_setup_hash(REQUESTED_RAW_SETUP)
EXPECTED_CACHE_PATH = rabi_prefix_cache_path(
    CACHE_DIR,
    RUN_PRESET,
    REQUESTED_RAW_SETUP,
    basename=RAW_PREFIX_BASENAME,
)


def find_legacy_high_fidelity_cache() -> tuple[Path, dict] | None:
    matches = []
    pattern = f"{RAW_PREFIX_BASENAME}_{RUN_PRESET}_*.npz"
    for path in sorted(CACHE_DIR.glob(pattern)):
        try:
            with np.load(path, allow_pickle=False) as payload:
                if "raw_setup" not in payload.files:
                    continue
                setup = json.loads(str(payload["raw_setup"]))
                required_diagnostics = all(
                    any(name.startswith(prefix) for name in payload.files)
                    for prefix in ("predictable_kl_h0__", "predictable_variance_h0__", "polarization_design_h1__")
                )
        except (OSError, ValueError, KeyError, json.JSONDecodeError):
            continue
        compatible = (
            setup.get("cache_version") == "prefix-raw-logb-v3-diagnostics"
            and setup.get("preset") == RUN_PRESET
            and np.isclose(float(setup.get("contrast", np.nan)), 0.999)
            and np.isclose(float(setup.get("readout_error", np.nan)), 0.0005)
            and setup.get("adaptive_sampler_strategy") == "periodic_hybrid_sphere_continuous_8"
            and required_diagnostics
        )
        if compatible:
            matches.append((path.resolve(), setup))
    if not matches:
        return None
    if len(matches) > 1:
        raise RuntimeError(f"multiple compatible legacy high-fidelity caches found: {[str(path) for path, _ in matches]}")
    return matches[0]

raw_cache_override = os.environ.get("RABI_DISCOVERY_RAW_CACHE", "").strip()
EXPECTED_RAW_SETUP = REQUESTED_RAW_SETUP
USING_LEGACY_HIGH_FIDELITY_CACHE = False
RAW_CACHE_PATH = (
    Path(raw_cache_override).expanduser().resolve()
    if raw_cache_override
    else EXPECTED_CACHE_PATH.resolve()
)
if not raw_cache_override and not RAW_CACHE_PATH.exists() and DETECTOR_PROFILE == "high_fidelity":
    legacy_match = find_legacy_high_fidelity_cache()
    if legacy_match is not None:
        RAW_CACHE_PATH, EXPECTED_RAW_SETUP = legacy_match
        USING_LEGACY_HIGH_FIDELITY_CACHE = True
RAW_SETUP_HASH = stable_setup_hash(EXPECTED_RAW_SETUP)
if not RAW_CACHE_PATH.exists():
    raise FileNotFoundError(
        "No raw prefix cache exactly matches the requested raw_setup.\n"
        f"expected_hash={REQUESTED_RAW_SETUP_HASH}\n"
        f"expected_path={EXPECTED_CACHE_PATH}\n"
        "Generate it with rabi_prefix_raw_data.ipynb using the same "
        "RABI_PREFIX_PRESET and RABI_PREFIX_* overrides."
    )

# Include a readable setup tag and the hash in every figure/table filename.
contrast_token = format(float(EXPECTED_RAW_SETUP["contrast"]), ".6g").replace("-", "m").replace(".", "p")
default_output_tag = f"C{contrast_token}_{RAW_SETUP_HASH[:8]}"
if SMALL_CHECK:
    default_output_tag = f"check_{default_output_tag}"
OUTPUT_TAG = os.environ.get("RABI_DISCOVERY_OUTPUT_TAG", default_output_tag).strip()
OUTPUT_STEM = f"{OUTPUT_BASENAME}_{RUN_PRESET}_{OUTPUT_TAG}"

z_text = os.environ.get("RABI_DISCOVERY_Z_STARS", "1.645")
Z_STARS = parse_float_list(z_text)
alpha_text = os.environ.get("RABI_DISCOVERY_ALPHA_LIST", "").strip()
ALPHAS = parse_float_list(alpha_text) if alpha_text else norm.sf(Z_STARS)
if ALPHAS.size != Z_STARS.size:
    Z_STARS = norm.isf(ALPHAS)

TARGET_POWERS = tuple(
    parse_float_list(os.environ.get("RABI_DISCOVERY_TARGET_POWERS", "0.5,0.7,0.9,0.95")).tolist()
)
DISCOVERY_N_SHOTS_TEXT = os.environ.get("RABI_DISCOVERY_N_SHOTS", "").strip()
SENSITIVITY_ERROR_DRAWS = int(os.environ.get("RABI_DISCOVERY_BOOTSTRAP_DRAWS", "4000"))
POWER_CURVE_MAX_CHECKPOINTS = int(os.environ.get("RABI_DISCOVERY_POWER_MAX_CHECKPOINTS", "8"))
FPOL_PHASES = parse_float_list(
    os.environ.get("RABI_DISCOVERY_FPOL_PHASES", "0.05,0.08,0.12,0.19")
)
COLLAPSE_SHOTS_TEXT = os.environ.get("RABI_DISCOVERY_COLLAPSE_SHOTS", "").strip()
ADAPTIVE_FIT_R2_MIN = float(os.environ.get("RABI_DISCOVERY_ADAPTIVE_FIT_R2_MIN", "0.985"))
ADAPTIVE_FIT_MIN_POINTS = int(os.environ.get("RABI_DISCOVERY_ADAPTIVE_FIT_MIN_POINTS", "5"))

print(json.dumps({
    "raw_cache_path": str(RAW_CACHE_PATH),
    "raw_setup_hash": RAW_SETUP_HASH,
    "output_tag": OUTPUT_TAG,
    "preset": RUN_PRESET,
    "detector_profile": EXPECTED_RAW_SETUP.get("detector_profile", DETECTOR_PROFILE),
    "expected_contrast": EXPECTED_RAW_SETUP["contrast"],
    "legacy_high_fidelity_cache": USING_LEGACY_HIGH_FIDELITY_CACHE,
    "z_stars": Z_STARS.tolist(),
    "alphas": ALPHAS.tolist(),
    "target_powers": list(TARGET_POWERS),
    "bootstrap_draws": SENSITIVITY_ERROR_DRAWS,
    "power_curve_max_checkpoints": POWER_CURVE_MAX_CHECKPOINTS,
    "fpol_phases": FPOL_PHASES.tolist(),
    "collapse_shots": COLLAPSE_SHOTS_TEXT or "auto",
    "adaptive_fit_r2_min": ADAPTIVE_FIT_R2_MIN,
    "adaptive_fit_min_points": ADAPTIVE_FIT_MIN_POINTS,
}, indent=2))


In [ ]:
def load_raw_prefix_cache(path: Path) -> dict:
    payload = np.load(path, allow_pickle=True)
    strategies = [str(item) for item in payload["strategies"].tolist()]
    diagnostic_prefixes = (
        "predictable_kl_h0",
        "predictable_variance_h0",
        "polarization_design_h0",
        "polarization_design_h1",
    )
    missing = [
        f"{prefix}__{strategy}"
        for strategy in strategies
        for prefix in diagnostic_prefixes
        if f"{prefix}__{strategy}" not in payload.files
    ]
    if missing:
        raise ValueError(
            "The selected raw cache does not contain the v3 trajectory diagnostics. "
            "Regenerate it with rabi_prefix_raw_data.ipynb. Missing keys: "
            + ", ".join(missing[:6])
        )
    raw = {
        "raw_setup": json.loads(str(payload["raw_setup"])),
        "checkpoints": np.asarray(payload["checkpoints"], dtype=np.int64),
        "phase_grid": np.asarray(payload["phase_grid"], dtype=np.float64),
        "strategies": strategies,
        "logB_h0": {},
        "logB_h1": {},
        "true_drive_phases_h1": {},
        "predictable_kl_h0": {},
        "predictable_variance_h0": {},
        "polarization_design_h0": {},
        "polarization_design_h1": {},
    }
    for strategy in strategies:
        raw["logB_h0"][strategy] = np.asarray(
            payload[f"logB_h0__{strategy}"], dtype=np.float64
        )
        raw["logB_h1"][strategy] = np.asarray(
            payload[f"logB_h1__{strategy}"], dtype=np.float64
        )
        raw["true_drive_phases_h1"][strategy] = np.asarray(
            payload[f"true_drive_phases_h1__{strategy}"], dtype=np.float64
        )
        raw["predictable_kl_h0"][strategy] = np.asarray(
            payload[f"predictable_kl_h0__{strategy}"], dtype=np.float64
        )
        raw["predictable_variance_h0"][strategy] = np.asarray(
            payload[f"predictable_variance_h0__{strategy}"], dtype=np.float64
        )
        raw["polarization_design_h0"][strategy] = np.asarray(
            payload[f"polarization_design_h0__{strategy}"], dtype=np.float64
        )
        raw["polarization_design_h1"][strategy] = np.asarray(
            payload[f"polarization_design_h1__{strategy}"], dtype=np.float64
        )
    return raw


raw = load_raw_prefix_cache(RAW_CACHE_PATH)
raw_setup = raw["raw_setup"]
assert_matching_raw_setup(EXPECTED_RAW_SETUP, raw_setup)
print(f"validated raw_setup hash {RAW_SETUP_HASH}")
CHECKPOINTS = raw["checkpoints"]
PHASE_GRID = raw["phase_grid"]
RAW_STRATEGIES = raw["strategies"]
STRATEGIES = [strategy for strategy in RAW_STRATEGIES if not strategy.startswith("adaptive_stein")]
ADAPTIVE_STRATEGIES = [
    strategy for strategy in STRATEGIES
    if strategy == "adaptive" or strategy.startswith("adaptive_l")
]
if not ADAPTIVE_STRATEGIES:
    raise ValueError("The selected cache contains no adaptive strategy")
PRIMARY_ADAPTIVE_STRATEGY = "adaptive_l1" if "adaptive_l1" in ADAPTIVE_STRATEGIES else ADAPTIVE_STRATEGIES[0]
DISCOVERY_N_SHOTS = int(DISCOVERY_N_SHOTS_TEXT) if DISCOVERY_N_SHOTS_TEXT else int(CHECKPOINTS[-1])
if DISCOVERY_N_SHOTS not in set(CHECKPOINTS.astype(int)):
    raise ValueError(f"Requested n_shots={DISCOVERY_N_SHOTS} is not in cached checkpoints {CHECKPOINTS.tolist()}")

STYLE = {
    "fixed_z": {"label": r"Fixed $z$", "color": "#4C78A8", "marker": "o"},
    "fixed_x": {"label": r"Fixed $x$", "color": "#F58518", "marker": "D"},
    "two_axis": {"label": r"Fixed $x/y$", "color": "#B279A2", "marker": "v"},
    "adaptive": {"label": r"Adaptive $L=8$", "color": "#54A24B", "marker": "s"},
    "adaptive_l1": {"label": r"Adaptive $L=1$ (IG)", "color": "#54A24B", "marker": "s"},
    "adaptive_l2": {"label": r"Adaptive $L=2$", "color": "#E45756", "marker": "P"},
}
ORACLE_KEY = "oracle_known_signal"
ORACLE_STYLE = {"label": "Transverse oracle", "color": "black", "marker": "^"}

print("checkpoints:", CHECKPOINTS.tolist())
print("phase_grid:", PHASE_GRID.tolist())
print("plotted strategies:", STRATEGIES)
for strategy in RAW_STRATEGIES:
    print(
        strategy,
        "H0",
        raw["logB_h0"][strategy].shape,
        "H1",
        raw["logB_h1"][strategy].shape,
        "Q(H1)",
        raw["polarization_design_h1"][strategy].shape,
    )


In [ ]:
def sensitivity_crossing(phases: np.ndarray, powers: np.ndarray, target: float) -> float:
    phases = np.asarray(phases, dtype=float)
    monotone = np.maximum.accumulate(np.asarray(powers, dtype=float))
    if monotone[-1] < target:
        return float("nan")
    index = int(np.searchsorted(monotone, target, side="left"))
    if index == 0:
        return float(phases[0])
    x0, x1 = phases[index - 1], phases[index]
    y0, y1 = monotone[index - 1], monotone[index]
    if y1 <= y0:
        return float(x1)
    return float(x0 + (target - y0) * (x1 - x0) / (y1 - y0))


def power_and_error(samples: np.ndarray, threshold: float) -> tuple[float, float]:
    power = float(np.mean(samples >= threshold))
    se = float(np.sqrt(max(power * (1.0 - power), 0.0) / samples.size))
    return power, se


def reconstruct_config(raw_setup: dict, n_shots: int) -> RabiExperimentConfig:
    particle_keys = {field.name for field in fields(RabiParticleSpec)}
    particle_payload = {key: value for key, value in raw_setup["particle_spec"].items() if key in particle_keys}
    particle_spec = RabiParticleSpec(**particle_payload)
    readout = RabiReadoutConfig(
        contrast=float(raw_setup["contrast"]),
        readout_error=float(raw_setup["readout_error"]),
        t1=float(raw_setup["t1_time"]),
        t2=float(raw_setup["t2_time"]),
    )
    candidate_angle_count = int(raw_setup.get("candidate_angle_count", 16))
    candidate_polar_count = int(raw_setup.get("candidate_polar_count", 7))
    return RabiExperimentConfig(
        n_shots=int(n_shots),
        sensing_time=float(raw_setup["sensing_time"]),
        dead_time=float(raw_setup.get("dead_time", 0.0)),
        true_phase_amplitude=float(PHASE_GRID[0]),
        true_detuning=0.0,
        true_drive_phase=0.0,
        readout=readout,
        particles=particle_spec,
        candidate_angles=tuple(np.linspace(0.0, math.pi, candidate_angle_count, endpoint=False)),
        candidate_polars=tuple(np.linspace(0.0, math.pi, candidate_polar_count)),
        seed=12345,
    )


def build_power_dataframe(raw: dict, alphas: np.ndarray, z_stars: np.ndarray) -> pd.DataFrame:
    rows = []
    checkpoints = raw["checkpoints"]
    phase_grid = raw["phase_grid"]
    for alpha, z_star in zip(alphas, z_stars):
        for strategy in raw["strategies"]:
            null = raw["logB_h0"][strategy]
            h1 = raw["logB_h1"][strategy]
            thresholds = np.asarray([null_threshold(null[:, j], float(alpha)) for j in range(checkpoints.size)])
            empirical_alpha = np.asarray([np.mean(null[:, j] >= thresholds[j]) for j in range(checkpoints.size)])
            for checkpoint_index, n_shots in enumerate(checkpoints):
                threshold = float(thresholds[checkpoint_index])
                for phase_index, phi_r in enumerate(phase_grid):
                    power, se = power_and_error(h1[phase_index, :, checkpoint_index], threshold)
                    rows.append(
                        {
                            "alpha": float(alpha),
                            "z_star": float(z_star),
                            "n_shots": int(n_shots),
                            "strategy": strategy,
                            "phase_amplitude": float(phi_r),
                            "rho_sqrt_n": float(phi_r * math.sqrt(float(n_shots))),
                            "rho_z": float(phi_r * float(n_shots) ** 0.25),
                            "threshold": threshold,
                            "empirical_alpha": float(empirical_alpha[checkpoint_index]),
                            "power": power,
                            "power_se": se,
                        }
                    )
    return pd.DataFrame(rows)


def transverse_oracle_power(
    config: RabiExperimentConfig,
    phase_amplitudes: np.ndarray | float,
    alpha: float,
) -> np.ndarray | float:
    """Exact binomial power when the transverse signal direction is known."""
    phase_amplitudes = np.asarray(phase_amplitudes, dtype=float)
    probability_h0 = float(rabi_probability_one(
        phase_amplitude=0.0, detuning=config.true_detuning, drive_phase=0.0,
        start_time=0.0, sensing_time=config.sensing_time, axis="transverse",
        readout_angle=0.5 * math.pi, readout=config.readout,
    ))
    probability_h1 = np.asarray(rabi_probability_one(
        phase_amplitude=phase_amplitudes, detuning=config.true_detuning, drive_phase=0.0,
        start_time=0.0, sensing_time=config.sensing_time, axis="transverse",
        readout_angle=0.5 * math.pi, readout=config.readout,
    ), dtype=float)
    return binomial_neyman_pearson_power(
        config.n_shots, probability_h0, probability_h1, float(alpha)
    )


def build_oracle_power_dataframe(raw_setup: dict, alphas: np.ndarray, z_stars: np.ndarray, checkpoints: np.ndarray) -> pd.DataFrame:
    phase_grid_dense = np.linspace(max(0.001, 0.5 * float(PHASE_GRID.min())), float(PHASE_GRID.max()), 240)
    rows = []
    for alpha, z_star in zip(alphas, z_stars):
        for n_shots in checkpoints:
            config = reconstruct_config(raw_setup, int(n_shots))
            powers = np.asarray(transverse_oracle_power(config, phase_grid_dense, float(alpha)), dtype=float)
            for phi_r, power in zip(phase_grid_dense, powers):
                rows.append(
                    {
                        "alpha": float(alpha),
                        "z_star": float(z_star),
                        "n_shots": int(n_shots),
                        "strategy": ORACLE_KEY,
                        "phase_amplitude": float(phi_r),
                        "rho_sqrt_n": float(phi_r * math.sqrt(float(n_shots))),
                        "rho_z": float(phi_r * float(n_shots) ** 0.25),
                        "threshold": np.nan,
                        "empirical_alpha": np.nan,
                        "power": float(power),
                        "power_se": 0.0,
                    }
                )
    return pd.DataFrame(rows)


def build_sensitivity_dataframe(power_df: pd.DataFrame, targets: tuple[float, ...]) -> pd.DataFrame:
    rows = []
    grouped = power_df.groupby(["alpha", "z_star", "n_shots", "strategy"], sort=False)
    for (alpha, z_star, n_shots, strategy), group in grouped:
        group = group.sort_values("phase_amplitude")
        phases = group["phase_amplitude"].to_numpy(dtype=float)
        powers = group["power"].to_numpy(dtype=float)
        for target in targets:
            rows.append(
                {
                    "alpha": float(alpha),
                    "z_star": float(z_star),
                    "n_shots": int(n_shots),
                    "strategy": strategy,
                    "target_power": float(target),
                    "phase_sensitivity": sensitivity_crossing(phases, powers, float(target)),
                }
            )
    return pd.DataFrame(rows)


def selected_plot_checkpoints(checkpoints: np.ndarray, max_count: int = 8) -> np.ndarray:
    checkpoints = np.asarray(checkpoints, dtype=int)
    if checkpoints.size <= max_count:
        return checkpoints
    indices = np.unique(np.round(np.linspace(0, checkpoints.size - 1, max_count)).astype(int))
    return checkpoints[indices]


def null_calibration_tolerance(alpha: float, sample_count: int) -> float:
    return max(0.02, 4.0 * math.sqrt(float(alpha) * (1.0 - float(alpha)) / max(int(sample_count), 1)))


def null_calibration_mask(rows: pd.DataFrame, alpha: float, sample_count: int) -> np.ndarray:
    if "empirical_alpha" not in rows:
        return np.ones(len(rows), dtype=bool)
    empirical_alpha = rows["empirical_alpha"].to_numpy(dtype=float)
    tolerance = null_calibration_tolerance(float(alpha), int(sample_count))
    return np.isfinite(empirical_alpha) & (np.abs(empirical_alpha - float(alpha)) <= tolerance)



def solve_increasing_target(function, target: float, initial_upper: float = 1.0) -> float:
    lower = 0.0
    upper = float(initial_upper)
    while function(upper) < target and upper < 100.0:
        upper *= 2.0
    if function(upper) < target:
        return float("nan")
    return float(brentq(lambda value: function(value) - target, lower, upper, xtol=1.0e-10))


def detector_response_coefficients(raw_setup: dict) -> dict:
    sensing_time = float(raw_setup["sensing_time"])
    t1_time = float(raw_setup["t1_time"])
    t2_time = float(raw_setup["t2_time"])
    contrast = float(raw_setup["contrast"])
    readout_error = float(raw_setup["readout_error"])
    readout_scale = 1.0 - 2.0 * readout_error
    eta_1 = longitudinal_relaxation_factor(sensing_time, t1_time, t2_time)
    eta_2 = transverse_relaxation_factor(sensing_time, t2_time)
    p_z0 = readout_error + 0.5 * readout_scale * (1.0 - contrast)
    b_transverse = 0.5 * readout_scale * contrast * eta_2
    quadratic_response = 0.25 * readout_scale * contrast * eta_1
    return {
        "readout_scale": readout_scale,
        "eta_1": eta_1,
        "eta_2": eta_2,
        "p_z0": p_z0,
        "b_transverse": b_transverse,
        "quadratic_response": quadratic_response,
    }


def fixed_x_required_local_snr(alpha: float, target: float) -> float:
    critical = float(norm.ppf(1.0 - 0.5 * alpha))
    phases = np.linspace(0.0, 2.0 * np.pi, 8192, endpoint=False)
    phase_projection = np.sin(phases)

    def phase_averaged_power(local_snr: float) -> float:
        means = local_snr * phase_projection
        conditional = norm.sf(critical - means) + norm.cdf(-critical - means)
        return float(np.mean(conditional))

    return solve_increasing_target(phase_averaged_power, target, initial_upper=2.0)


def two_axis_required_noncentrality(alpha: float, target: float) -> float:
    critical = float(chi2.ppf(1.0 - alpha, df=2))
    return solve_increasing_target(
        lambda noncentrality: float(ncx2.sf(critical, df=2, nc=noncentrality)),
        target,
        initial_upper=4.0,
    )


def randomized_poisson_upper_tail_power(mu0: float, mu1: float, alpha: float) -> float:
    boundary = max(0, int(poisson.isf(alpha, mu0)))
    strict_null_tail = float(poisson.sf(boundary, mu0))
    boundary_null_mass = float(poisson.pmf(boundary, mu0))
    randomization = (
        np.clip((alpha - strict_null_tail) / boundary_null_mass, 0.0, 1.0)
        if boundary_null_mass > 0.0
        else 0.0
    )
    return float(poisson.sf(boundary, mu1) + randomization * poisson.pmf(boundary, mu1))


def randomized_two_sided_binomial_power(
    n_shots: int,
    probability_h1: np.ndarray,
    alpha: float,
) -> np.ndarray:
    """Two-sided count-test power for the transverse H0 probability 1/2."""
    probability_h0 = 0.5
    lower_boundary = int(binom.ppf(0.5 * alpha, n_shots, probability_h0))
    upper_boundary = n_shots - lower_boundary
    strict_null_tail = float(
        binom.cdf(lower_boundary - 1, n_shots, probability_h0)
        + binom.sf(upper_boundary, n_shots, probability_h0)
    )
    boundary_null_mass = float(
        binom.pmf(lower_boundary, n_shots, probability_h0)
        + binom.pmf(upper_boundary, n_shots, probability_h0)
    )
    randomization = (
        np.clip((alpha - strict_null_tail) / boundary_null_mass, 0.0, 1.0)
        if boundary_null_mass > 0.0
        else 0.0
    )
    probability_h1 = np.asarray(probability_h1, dtype=float)
    return (
        binom.cdf(lower_boundary - 1, n_shots, probability_h1)
        + binom.sf(upper_boundary, n_shots, probability_h1)
        + randomization
        * (
            binom.pmf(lower_boundary, n_shots, probability_h1)
            + binom.pmf(upper_boundary, n_shots, probability_h1)
        )
    )


def fixed_strategy_theory_power(
    strategy: str,
    config: RabiExperimentConfig,
    phase_amplitudes: np.ndarray,
    alpha: float,
    drive_phase_count: int = 2048,
) -> np.ndarray:
    """Finite-Rabi theory curves, averaged over the unknown drive phase."""
    phase_amplitudes = np.asarray(phase_amplitudes, dtype=float)
    if strategy == "fixed_z":
        probability_h0 = float(rabi_probability_one(
            phase_amplitude=0.0, detuning=0.0, drive_phase=0.0, start_time=0.0,
            sensing_time=config.sensing_time, axis="z", readout_angle=0.0,
            readout=config.readout,
        ))
        probability_h1 = np.asarray(rabi_probability_one(
            phase_amplitude=phase_amplitudes, detuning=0.0, drive_phase=0.0, start_time=0.0,
            sensing_time=config.sensing_time, axis="z", readout_angle=0.0,
            readout=config.readout,
        ), dtype=float)
        return np.asarray(binomial_neyman_pearson_power(
            config.n_shots, probability_h0, probability_h1, alpha
        ), dtype=float)

    drive_phases = np.linspace(0.0, 2.0 * np.pi, int(drive_phase_count), endpoint=False)
    amplitude_mesh = phase_amplitudes[:, None]
    phase_mesh = drive_phases[None, :]
    probability_h0 = 0.5
    probability_x = np.asarray(rabi_probability_one(
        phase_amplitude=amplitude_mesh, detuning=0.0, drive_phase=phase_mesh, start_time=0.0,
        sensing_time=config.sensing_time, axis="transverse", readout_angle=0.0,
        readout=config.readout,
    ), dtype=float)
    if strategy == "fixed_x":
        conditional_power = randomized_two_sided_binomial_power(
            config.n_shots, probability_x, alpha
        )
        return np.mean(conditional_power, axis=1)

    if strategy == "two_axis":
        probability_y = np.asarray(rabi_probability_one(
            phase_amplitude=amplitude_mesh, detuning=0.0, drive_phase=phase_mesh, start_time=0.0,
            sensing_time=config.sensing_time, axis="transverse", readout_angle=0.5 * math.pi,
            readout=config.readout,
        ), dtype=float)
        n_x = (config.n_shots + 1) // 2
        n_y = config.n_shots // 2
        variance_h0 = probability_h0 * (1.0 - probability_h0)
        noncentrality = (
            n_x * (probability_x - probability_h0) ** 2
            + n_y * (probability_y - probability_h0) ** 2
        ) / variance_h0
        critical = float(chi2.ppf(1.0 - alpha, df=2))
        return np.mean(ncx2.sf(critical, df=2, nc=noncentrality), axis=1)

    raise ValueError(f"no fixed-strategy theory curve implemented for {strategy!r}")


def fixed_z_poisson_sensitivity(
    n_shots: float,
    alpha: float,
    target: float,
    response: dict,
) -> float:
    mu0 = float(n_shots) * response["p_z0"]

    def power(phase_amplitude: float) -> float:
        mu1 = float(n_shots) * (
            response["p_z0"] + response["quadratic_response"] * phase_amplitude**2
        )
        return randomized_poisson_upper_tail_power(mu0, mu1, alpha)

    return solve_increasing_target(power, target, initial_upper=0.1)


def analytic_sensitivity(
    strategy: str,
    n_values: np.ndarray,
    alpha: float,
    target: float,
    response: dict,
) -> tuple[np.ndarray, str]:
    n_values = np.asarray(n_values, dtype=float)
    if strategy == "fixed_z":
        gaussian_threshold = norm.ppf(1.0 - alpha) + norm.ppf(target)
        coefficient = math.sqrt(
            gaussian_threshold
            * math.sqrt(response["p_z0"] * (1.0 - response["p_z0"]))
            / response["quadratic_response"]
        )
        return coefficient * n_values**-0.25, rf"weak-signal asymptotic, $A={coefficient:.2f}$"
    if strategy == "fixed_x":
        local_snr = fixed_x_required_local_snr(alpha, target)
        coefficient = local_snr / (2.0 * response["b_transverse"])
        return coefficient * n_values**-0.5, rf"weak-signal analytic, $A={coefficient:.2f}$"
    if strategy == "two_axis":
        noncentrality = two_axis_required_noncentrality(alpha, target)
        coefficient = math.sqrt(noncentrality / (2.0 * response["b_transverse"] ** 2))
        return coefficient * n_values**-0.5, rf"weak-signal analytic, $A={coefficient:.2f}$"
    if strategy == ORACLE_KEY:
        coefficient = (
            norm.ppf(1.0 - alpha) + norm.ppf(target)
        ) / (2.0 * response["b_transverse"])
        return coefficient * n_values**-0.5, rf"weak-signal oracle, $A={coefficient:.2f}$"
    raise ValueError(f"no analytic sensitivity implemented for {strategy!r}")


def log_log_fit(n_values: np.ndarray, sensitivities: np.ndarray) -> dict:
    x = np.log(np.asarray(n_values, dtype=float))
    y = np.log(np.asarray(sensitivities, dtype=float))
    slope, intercept = np.polyfit(x, y, 1)
    prediction = intercept + slope * x
    residual = float(np.sum((y - prediction) ** 2))
    total = float(np.sum((y - np.mean(y)) ** 2))
    r_squared = 1.0 - residual / total if total > 0.0 else 1.0
    return {"intercept": float(intercept), "slope": float(slope), "r_squared": float(r_squared)}


def adaptive_tail_power_law_fit(n_values: np.ndarray, sensitivities: np.ndarray) -> dict | None:
    finite = np.isfinite(sensitivities) & (sensitivities > 0.0) & (n_values > 0.0)
    n_finite = np.asarray(n_values[finite], dtype=float)
    sensitivity_finite = np.asarray(sensitivities[finite], dtype=float)
    if n_finite.size < ADAPTIVE_FIT_MIN_POINTS:
        return None

    last_start = n_finite.size - ADAPTIVE_FIT_MIN_POINTS
    first_allowed = min(n_finite.size // 2, last_start)
    candidates = []
    for start in range(first_allowed, last_start + 1):
        result = log_log_fit(n_finite[start:], sensitivity_finite[start:])
        result.update({
            "start": start,
            "n_values": n_finite[start:],
            "sensitivities": sensitivity_finite[start:],
        })
        candidates.append(result)

    if not candidates:
        return None

    acceptable = [item for item in candidates if item["r_squared"] >= ADAPTIVE_FIT_R2_MIN]
    return acceptable[0] if acceptable else max(
        candidates,
        key=lambda item: (item["r_squared"], -item["start"]),
    )


def sensitivity_crossing_draws(
    phases: np.ndarray,
    powers: np.ndarray,
    target: float,
    sample_count: int,
    draws: int,
    seed: int,
) -> np.ndarray:
    if draws <= 0:
        return np.empty(0, dtype=float)
    successes = np.rint(np.asarray(powers, dtype=float) * sample_count).astype(int)
    rng = np.random.default_rng(seed)
    sampled_powers = rng.beta(
        successes[None, :] + 0.5,
        sample_count - successes[None, :] + 0.5,
        size=(draws, successes.size),
    )
    sampled_powers = np.maximum.accumulate(sampled_powers, axis=1)
    return np.asarray(
        [sensitivity_crossing(phases, sampled, target) for sampled in sampled_powers],
        dtype=float,
    )


def sensitivity_crossing_interval(
    phases: np.ndarray,
    powers: np.ndarray,
    target: float,
    sample_count: int,
    draws: int,
    seed: int,
) -> tuple[float, float, float]:
    crossings = sensitivity_crossing_draws(
        phases, powers, target, sample_count, draws, seed
    )
    finite_crossings = crossings[np.isfinite(crossings)]
    if finite_crossings.size == 0:
        return float("nan"), float("nan"), 0.0
    lower, upper = np.quantile(finite_crossings, [0.16, 0.84])
    return float(lower), float(upper), float(finite_crossings.size / crossings.size)


In [ ]:
power_df = build_power_dataframe(raw, ALPHAS, Z_STARS)
oracle_power_df = build_oracle_power_dataframe(raw_setup, ALPHAS, Z_STARS, CHECKPOINTS)
combined_power_df = pd.concat([power_df, oracle_power_df], ignore_index=True)
sensitivity_df = build_sensitivity_dataframe(combined_power_df, TARGET_POWERS)

display(combined_power_df.head())
display(sensitivity_df.head())


In [ ]:
z_to_plot = float(Z_STARS[0])
alpha_to_plot = float(ALPHAS[0])
n_null = int(raw_setup.get("n_null", next(iter(raw["logB_h0"].values())).shape[0]))
plot_checkpoints = selected_plot_checkpoints(CHECKPOINTS, POWER_CURVE_MAX_CHECKPOINTS)
n_columns = min(4, plot_checkpoints.size)
n_rows = int(np.ceil(plot_checkpoints.size / n_columns))
fig, axes = plt.subplots(n_rows, n_columns, figsize=(3.8 * n_columns, 3.25 * n_rows), sharex=True, sharey=True)
axes = np.asarray(axes).ravel()

for ax, n_shots in zip(axes, plot_checkpoints):
    for strategy in STRATEGIES:
        rows = combined_power_df[
            (np.isclose(combined_power_df["z_star"], z_to_plot))
            & (combined_power_df["n_shots"] == int(n_shots))
            & (combined_power_df["strategy"] == strategy)
        ].sort_values("phase_amplitude")
        rows = rows.loc[null_calibration_mask(rows, alpha_to_plot, n_null)]
        if rows.empty:
            continue
        meta = STYLE.get(strategy, {"label": strategy, "color": None, "marker": "o"})
        ax.errorbar(
            rows["phase_amplitude"],
            rows["power"],
            yerr=rows["power_se"],
            marker=meta["marker"],
            linewidth=1.3,
            markersize=3.2,
            markerfacecolor="white",
            markeredgewidth=0.9,
            capsize=2.8,
            capthick=1.0,
            elinewidth=1.0,
            color=meta["color"],
            label=meta["label"],
        )
    oracle_rows = combined_power_df[
        (np.isclose(combined_power_df["z_star"], z_to_plot))
        & (combined_power_df["n_shots"] == int(n_shots))
        & (combined_power_df["strategy"] == ORACLE_KEY)
    ].sort_values("phase_amplitude")
    if not oracle_rows.empty:
        ax.plot(
            oracle_rows["phase_amplitude"],
            oracle_rows["power"],
            color=ORACLE_STYLE["color"],
            linestyle="--",
            linewidth=1.5,
            label=ORACLE_STYLE["label"],
        )
    for target in TARGET_POWERS:
        ax.axhline(target, color="0.35", linestyle=":", linewidth=0.85, alpha=0.75)
    ax.set_title(rf"$n_{{\rm shots}}={int(n_shots)}$")
    ax.set_xlabel(r"Rabi phase amplitude $\Phi_R$")
    ax.set_ylabel(r"Detection probability")
    ax.set_ylim(-0.02, 1.03)
    ax.legend(frameon=False, fontsize=7.5)

for ax in axes[plot_checkpoints.size:]:
    ax.axis("off")

fig.suptitle(rf"Power curves at $Z_\star={z_to_plot:.3g}$ ($\alpha={alpha_to_plot:.3g}$)", y=1.01)
fig.tight_layout()
pdf_path = FIGURE_DIR / f"{OUTPUT_STEM}_power_curves_z{z_to_plot:.3f}.pdf"
fig.savefig(pdf_path, bbox_inches="tight")
print(f"saved {pdf_path}")
plt.show()


In [ ]:
z_to_plot = float(Z_STARS[0])
alpha_to_plot = float(ALPHAS[0])
if COLLAPSE_SHOTS_TEXT:
    collapse_shots = parse_int_list(COLLAPSE_SHOTS_TEXT)
else:
    collapse_shots = selected_plot_checkpoints(CHECKPOINTS, max_count=4)
available_checkpoints = set(int(value) for value in CHECKPOINTS)
collapse_shots = np.asarray([int(n) for n in collapse_shots if int(n) in available_checkpoints], dtype=int)
if collapse_shots.size < 2:
    raise ValueError("the collapse plot requires at least two cached shot-count checkpoints")

collapse_colors = plt.cm.viridis(np.linspace(0.08, 0.88, collapse_shots.size))
collapse_color_by_n = {int(n): color for n, color in zip(collapse_shots, collapse_colors)}
collapse_panels = [
    ("fixed_x", r"Fixed $x$", 0.5, r"$\rho=\Phi_R\sqrt{n_{\rm shots}}$"),
    ("two_axis", r"Fixed $x/y$", 0.5, r"$\rho=\Phi_R\sqrt{n_{\rm shots}}$"),
    *[(strategy, STYLE[strategy]["label"], 0.5, r"$\rho=\Phi_R\sqrt{n_{\rm shots}}$") for strategy in ADAPTIVE_STRATEGIES],
    (ORACLE_KEY, "Transverse oracle", 0.5, r"$\rho=\Phi_R\sqrt{n_{\rm shots}}$"),
    ("fixed_z", r"Fixed $z$: natural scaling", 0.25, r"$\rho_z=\Phi_R n_{\rm shots}^{1/4}$"),
    ("fixed_z", r"Fixed $z$: transverse scaling", 0.5, r"$\rho=\Phi_R\sqrt{n_{\rm shots}}$"),
]

n_collapse_columns = 4
n_collapse_rows = int(np.ceil(len(collapse_panels) / n_collapse_columns))
fig, axes = plt.subplots(n_collapse_rows, n_collapse_columns, figsize=(4.4 * n_collapse_columns, 3.6 * n_collapse_rows), sharey=True)
axes = np.asarray(axes).ravel()
for ax, (strategy, title, exponent, xlabel) in zip(axes, collapse_panels):
    for n_shots in collapse_shots:
        rows = combined_power_df[
            (np.isclose(combined_power_df["z_star"], z_to_plot))
            & (combined_power_df["n_shots"] == int(n_shots))
            & (combined_power_df["strategy"] == strategy)
        ].sort_values("phase_amplitude")
        if strategy in STRATEGIES:
            rows = rows.loc[null_calibration_mask(rows, alpha_to_plot, n_null)]
        if rows.empty:
            continue
        scaled_signal = rows["phase_amplitude"].to_numpy(dtype=float) * float(n_shots) ** exponent
        color = collapse_color_by_n[int(n_shots)]
        label = rf"$n=2^{{{int(round(math.log2(n_shots)))}}}$" if n_shots > 0 else "$n=0$"
        if strategy == ORACLE_KEY:
            ax.plot(scaled_signal, rows["power"], color=color, linewidth=1.6, label=label)
        else:
            ax.errorbar(
                scaled_signal,
                rows["power"],
                yerr=rows["power_se"],
                color=color,
                marker="o",
                markersize=3.0,
                markerfacecolor="white",
                markeredgewidth=0.8,
                capsize=2.4,
                linewidth=1.1,
                label=label,
            )
    for target in TARGET_POWERS:
        ax.axhline(target, color="0.35", linestyle=":", linewidth=0.8, alpha=0.7)
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(r"Detection probability")
    ax.set_ylim(-0.02, 1.03)
    ax.legend(frameon=False, fontsize=7.2)
for ax in axes[len(collapse_panels):]:
    ax.axis("off")

fig.suptitle(rf"Scaling collapse at $Z_\star={z_to_plot:.3g}$", y=1.01)
fig.tight_layout()
pdf_path = FIGURE_DIR / f"{OUTPUT_STEM}_power_data_collapse_z{z_to_plot:.3f}.pdf"
fig.savefig(pdf_path, bbox_inches="tight")
print(f"saved {pdf_path}")
plt.show()


In [ ]:
z_to_plot = float(Z_STARS[0])
alpha_to_plot = float(ALPHAS[0])
fixed_shot_plot_values = tuple(dict.fromkeys(
    ([1024] if 1024 in set(CHECKPOINTS.astype(int)) else []) + [DISCOVERY_N_SHOTS]
))
missing_fixed_shots = [n for n in fixed_shot_plot_values if n not in set(CHECKPOINTS.astype(int))]
if missing_fixed_shots:
    raise ValueError(f"Fixed-shot plot checkpoints are absent from the cache: {missing_fixed_shots}")

fixed_shot_adaptive_strategies = [PRIMARY_ADAPTIVE_STRATEGY]
missing_fixed_shot_adaptive = [
    strategy for strategy in fixed_shot_adaptive_strategies if strategy not in ADAPTIVE_STRATEGIES
]
if missing_fixed_shot_adaptive:
    raise ValueError(f"Missing fixed-shot adaptive strategies: {missing_fixed_shot_adaptive}")

for n_shots_to_plot in fixed_shot_plot_values:
    plot_rows = combined_power_df[
        (np.isclose(combined_power_df["z_star"], z_to_plot))
        & (combined_power_df["n_shots"] == int(n_shots_to_plot))
    ]
    config = reconstruct_config(raw_setup, int(n_shots_to_plot))
    rho_theory = np.linspace(0.0, 20.0, 360)
    phase_theory = rho_theory / math.sqrt(float(n_shots_to_plot))

    fig, ax = plt.subplots(figsize=(7.2, 5.3))
    fixed_shot_strategy_order = [*fixed_shot_adaptive_strategies, "fixed_z", "two_axis", "fixed_x"]
    for strategy in fixed_shot_strategy_order:
        rows = plot_rows[plot_rows["strategy"] == strategy].sort_values("rho_sqrt_n")
        meta = STYLE.get(strategy, {"label": strategy, "color": None, "marker": "o"})
        is_fixed = strategy in {"fixed_z", "fixed_x", "two_axis"}
        ax.errorbar(
            rows["rho_sqrt_n"],
            rows["power"],
            yerr=rows["power_se"],
            marker=meta["marker"],
            linestyle="none",
            linewidth=1.9,
            markersize=5.2,
            markerfacecolor="white",
            markeredgewidth=1.2,
            capsize=3.2,
            elinewidth=1.2,
            color=meta["color"],
            label="_nolegend_" if is_fixed else meta["label"],
            zorder=4,
        )
        if is_fixed:
            theory_power = fixed_strategy_theory_power(
                strategy, config, phase_theory, alpha_to_plot
            )
            ax.plot(
                rho_theory, theory_power,
                color=meta["color"], linestyle=":", linewidth=2.2,
                label=meta["label"], zorder=3,
            )

    oracle_power = np.asarray(
        transverse_oracle_power(config, phase_theory, alpha_to_plot), dtype=float
    )
    ax.plot(
        rho_theory, oracle_power,
        color=ORACLE_STYLE["color"], linestyle="--", linewidth=2.2,
        label=ORACLE_STYLE["label"], zorder=2,
    )

    for target in [1.0]:
        ax.axhline(target, color="0.35", linestyle=":", linewidth=1.0, alpha=0.78, zorder=0)
    ax.set_xlabel(r"Scaled signal strength $\rho$", fontsize=17)
    ax.set_ylabel(r"Phase-averaged power $\overline{P}$", fontsize=17)
    ax.set_xlim(0.0, 20.0)
    ax.set_xticks([0, 5, 10, 15, 20])
    ax.set_ylim(0.0, 1.03)
    ax.grid(False, which="both")
    ax.tick_params(axis="both", which="major", labelsize=15, direction="in", length=5.5, top=True, right=True)
    legend_labels = [
        *[STYLE[strategy]["label"] for strategy in fixed_shot_adaptive_strategies],
        STYLE["two_axis"]["label"], STYLE["fixed_x"]["label"],
        STYLE["fixed_z"]["label"], ORACLE_STYLE["label"],
    ]
    legend_handles, discovered_labels = ax.get_legend_handles_labels()
    legend_lookup = dict(zip(discovered_labels, legend_handles))
    ax.legend(
        [legend_lookup[label] for label in legend_labels], legend_labels,
        frameon=False, fontsize=13.2, loc="lower right",
    )
    fig.tight_layout()

    pdf_path = FIGURE_DIR / f"{OUTPUT_STEM}_discovery_n{int(n_shots_to_plot)}_z{z_to_plot:.3f}.pdf"
    fig.savefig(pdf_path, bbox_inches="tight")
    print(f"saved {pdf_path}")
    plt.show()


In [ ]:
z_to_plot = float(Z_STARS[0])
alpha_to_plot = float(ALPHAS[0])
rows_for_z = sensitivity_df[np.isclose(sensitivity_df["z_star"], z_to_plot)]
response = detector_response_coefficients(raw_setup)
n_signal = int(raw_setup.get("n_signal", next(iter(raw["logB_h1"].values())).shape[1]))
n_null = int(raw_setup.get("n_null", next(iter(raw["logB_h0"].values())).shape[0]))

# Reconstruct uncertainty on the interpolated sensitivity from the finite H1 MC sample.
error_rows = []
for target_index, target in enumerate(TARGET_POWERS):
    for strategy_index, strategy in enumerate(STRATEGIES):
        for checkpoint_index, n_shots in enumerate(CHECKPOINTS):
            rows = combined_power_df[
                (np.isclose(combined_power_df["z_star"], z_to_plot))
                & (combined_power_df["strategy"] == strategy)
                & (combined_power_df["n_shots"] == int(n_shots))
            ].sort_values("phase_amplitude")
            lower, upper, finite_fraction = sensitivity_crossing_interval(
                rows["phase_amplitude"].to_numpy(dtype=float),
                rows["power"].to_numpy(dtype=float),
                float(target),
                n_signal,
                SENSITIVITY_ERROR_DRAWS,
                seed=83_000 + 10_000 * target_index + 1_000 * strategy_index + checkpoint_index,
            )
            error_rows.append({
                "z_star": z_to_plot,
                "n_shots": int(n_shots),
                "strategy": strategy,
                "target_power": float(target),
                "phase_sensitivity_lower": lower,
                "phase_sensitivity_upper": upper,
                "crossing_draw_fraction": finite_fraction,
            })

sensitivity_error_df = pd.DataFrame(error_rows)
sensitivity_plot_df = rows_for_z.merge(
    sensitivity_error_df,
    on=["z_star", "n_shots", "strategy", "target_power"],
    how="left",
)

# Omit MC points whose discrete null threshold is visibly miscalibrated.
calibration_df = (
    combined_power_df[
        np.isclose(combined_power_df["z_star"], z_to_plot)
        & combined_power_df["strategy"].isin(STRATEGIES)
    ]
    .groupby(["n_shots", "strategy"], as_index=False)["empirical_alpha"]
    .first()
)
sensitivity_plot_df = sensitivity_plot_df.merge(
    calibration_df,
    on=["n_shots", "strategy"],
    how="left",
)
calibration_tolerance = null_calibration_tolerance(alpha_to_plot, n_null)
uncalibrated = (
    sensitivity_plot_df["strategy"].isin(STRATEGIES)
    & (
        ~np.isfinite(sensitivity_plot_df["empirical_alpha"])
        | (np.abs(sensitivity_plot_df["empirical_alpha"] - alpha_to_plot) > calibration_tolerance)
    )
)
sensitivity_plot_df.loc[
    uncalibrated,
    ["phase_sensitivity", "phase_sensitivity_lower", "phase_sensitivity_upper"],
] = np.nan

fig, axes = plt.subplots(
    1,
    len(TARGET_POWERS),
    figsize=(5.2 * len(TARGET_POWERS), 4.8),
    sharex=True,
    sharey=True,
)
axes = np.atleast_1d(axes)
fit_rows = []
analytic_rows = []
all_styles = {**STYLE, ORACLE_KEY: ORACLE_STYLE}

for ax, target in zip(axes, TARGET_POWERS):
    for strategy in [*STRATEGIES, ORACLE_KEY]:
        meta = all_styles[strategy]
        rows = sensitivity_plot_df[
            (sensitivity_plot_df["strategy"] == strategy)
            & np.isclose(sensitivity_plot_df["target_power"], float(target))
        ].sort_values("n_shots")
        n_values = rows["n_shots"].to_numpy(dtype=float)
        sensitivities = rows["phase_sensitivity"].to_numpy(dtype=float)
        finite = np.isfinite(sensitivities) & (sensitivities > 0.0)

        if strategy == ORACLE_KEY:
            ax.plot(
                n_values[finite],
                sensitivities[finite],
                marker=meta["marker"],
                linestyle="none",
                markersize=4.0,
                markerfacecolor="white",
                markeredgewidth=1.0,
                color=meta["color"],
                alpha=0.85,
                zorder=4,
                label=rf"{meta['label']} exact",
            )
        else:
            lower = rows["phase_sensitivity_lower"].to_numpy(dtype=float)
            upper = rows["phase_sensitivity_upper"].to_numpy(dtype=float)
            error_finite = finite & np.isfinite(lower) & np.isfinite(upper)
            plain_finite = finite & ~error_finite
            if np.any(error_finite):
                yerr = np.vstack([
                    np.maximum(sensitivities[error_finite] - lower[error_finite], 0.0),
                    np.maximum(upper[error_finite] - sensitivities[error_finite], 0.0),
                ])
                ax.errorbar(
                    n_values[error_finite],
                    sensitivities[error_finite],
                    yerr=yerr,
                    marker=meta["marker"],
                    linestyle="none",
                    markersize=3.2,
                    markerfacecolor="white",
                    markeredgewidth=0.95,
                    capsize=3.0,
                    capthick=1.15,
                    elinewidth=1.15,
                    color=meta["color"],
                    alpha=0.98,
                    zorder=5,
                    label=rf"{meta['label']} MC",
                )
            if np.any(plain_finite):
                ax.plot(
                    n_values[plain_finite],
                    sensitivities[plain_finite],
                    marker=meta["marker"],
                    linestyle="none",
                    markersize=3.2,
                    markerfacecolor="white",
                    markeredgewidth=0.95,
                    color=meta["color"],
                    alpha=0.98,
                    zorder=5,
                    label=rf"{meta['label']} MC" if not np.any(error_finite) else None,
                )

        if not strategy.startswith("adaptive"):
            finite_n = n_values[np.isfinite(n_values) & (n_values > 0.0)]
            if finite_n.size == 0:
                continue
            dense_n = np.geomspace(float(np.min(finite_n)), float(np.max(finite_n)), 300)
            analytic_values, description = analytic_sensitivity(
                strategy,
                dense_n,
                alpha_to_plot,
                float(target),
                response,
            )
            analytic_label = (
                rf"{meta['label']} asymptotic $n^{{-1/4}}$"
                if strategy == "fixed_z"
                else rf"{meta['label']} analytic"
            )
            analytic_linestyle = ":" if strategy in {"fixed_x", "two_axis"} else "--"
            ax.plot(
                dense_n,
                analytic_values,
                color=meta["color"],
                linestyle=analytic_linestyle,
                linewidth=1.6,
                alpha=0.9,
                label=analytic_label,
            )
            analytic_rows.append({
                "z_star": z_to_plot,
                "target_power": float(target),
                "strategy": strategy,
                "description": description,
            })
            if strategy == "fixed_z":
                poisson_values = np.asarray([
                    fixed_z_poisson_sensitivity(
                        n_shots,
                        alpha_to_plot,
                        float(target),
                        response,
                    )
                    for n_shots in dense_n
                ])
                ax.plot(
                    dense_n,
                    poisson_values,
                    color=meta["color"],
                    linestyle=":",
                    linewidth=1.7,
                    alpha=0.98,
                    label=rf"{meta['label']} Poisson",
                )
            continue

        fitted = adaptive_tail_power_law_fit(n_values, sensitivities)
        if fitted is None:
            continue
        fit_n = np.geomspace(float(fitted["n_values"][0]), float(fitted["n_values"][-1]), 160)
        fit_values = np.exp(fitted["intercept"]) * fit_n ** fitted["slope"]
        ax.plot(
            fit_n,
            fit_values,
            color=meta["color"],
            linestyle="--",
            linewidth=1.8,
            label=rf"{meta['label']} tail fit $s={fitted['slope']:.2f}$, $R^2={fitted['r_squared']:.3f}$",
        )
        fit_rows.append({
            "z_star": z_to_plot,
            "target_power": float(target),
            "strategy": strategy,
            "fitted_slope": fitted["slope"],
            "intercept": fitted["intercept"],
            "r_squared": fitted["r_squared"],
            "n_fit_min": float(fitted["n_values"][0]),
            "n_fit_max": float(fitted["n_values"][-1]),
            "n_fit_points": int(fitted["n_values"].size),
        })

    ax.set_xscale("log", base=2)
    ax.set_yscale("log")
    ax.set_xlabel(r"Number of shots $n$")
    ax.set_ylabel(r"Required signal phase $\Phi_R$")
    ax.set_title(rf"$P_{{\rm disc}}={float(target):.2g}$ at $Z_\star={z_to_plot:.3g}$")
    ax.legend(frameon=False, fontsize=7.2)

fig.tight_layout()
pdf_path = FIGURE_DIR / f"{OUTPUT_STEM}_sensitivity_z{z_to_plot:.3f}.pdf"
fig.savefig(pdf_path, bbox_inches="tight")
print(f"saved {pdf_path}")
plt.show()

sensitivity_fit_df = pd.DataFrame(fit_rows, columns=[
    "z_star", "target_power", "strategy", "fitted_slope", "intercept",
    "r_squared", "n_fit_min", "n_fit_max", "n_fit_points",
])
sensitivity_analytic_df = pd.DataFrame(analytic_rows)
display(sensitivity_fit_df)
display(sensitivity_analytic_df)


In [ ]:
single_target = 0.7
if not any(np.isclose(single_target, target) for target in TARGET_POWERS):
    raise ValueError(f"P_H1={single_target} is absent from TARGET_POWERS={TARGET_POWERS}")

fig, ax = plt.subplots(figsize=(7.2, 5.3))
single_adaptive_strategies = [PRIMARY_ADAPTIVE_STRATEGY]
missing_single_strategies = [
    strategy for strategy in single_adaptive_strategies if strategy not in ADAPTIVE_STRATEGIES
]
if missing_single_strategies:
    raise ValueError(f"Missing production adaptive strategies: {missing_single_strategies}")
single_plot_order = [*single_adaptive_strategies, "fixed_z", "two_axis", ORACLE_KEY]
single_legend_handles = {}
adaptive_fit_labels = {}
all_styles = {**STYLE, ORACLE_KEY: ORACLE_STYLE}
single_n_min = max(2.0**6, float(CHECKPOINTS.min()))
single_n_max = float(CHECKPOINTS.max())

for strategy in single_plot_order:
    meta = all_styles[strategy]
    rows = sensitivity_plot_df[
        (sensitivity_plot_df["strategy"] == strategy)
        & np.isclose(sensitivity_plot_df["target_power"], single_target)
    ].sort_values("n_shots")
    n_values = rows["n_shots"].to_numpy(dtype=float)
    sensitivities = rows["phase_sensitivity"].to_numpy(dtype=float)
    finite = np.isfinite(sensitivities) & (sensitivities > 0.0)

    if strategy != ORACLE_KEY:
        lower = rows["phase_sensitivity_lower"].to_numpy(dtype=float)
        upper = rows["phase_sensitivity_upper"].to_numpy(dtype=float)
        error_finite = finite & np.isfinite(lower) & np.isfinite(upper)
        if np.any(error_finite):
            yerr = np.vstack([
                np.maximum(sensitivities[error_finite] - lower[error_finite], 0.0),
                np.maximum(upper[error_finite] - sensitivities[error_finite], 0.0),
            ])
            point_handle = ax.errorbar(
                n_values[error_finite], sensitivities[error_finite], yerr=yerr,
                marker=meta["marker"], linestyle="none", markersize=4.6,
                markerfacecolor="white", markeredgewidth=1.1, capsize=3.2,
                capthick=1.2, elinewidth=1.2, color=meta["color"], zorder=5,
            )
            if strategy in single_adaptive_strategies:
                single_legend_handles[strategy] = point_handle

    dense_n = np.geomspace(single_n_min, single_n_max, 320)
    if strategy in {"fixed_z", "fixed_x", "two_axis", ORACLE_KEY}:
        analytic_values, _ = analytic_sensitivity(
            strategy, dense_n, alpha_to_plot, single_target, response
        )
        analytic_style = "--" if strategy in {"fixed_z", ORACLE_KEY} else ":"
        analytic_line, = ax.plot(
            dense_n, analytic_values, color=meta["color"],
            linestyle=analytic_style, linewidth=2.0, alpha=0.94, zorder=3,
        )
        single_legend_handles[strategy] = analytic_line
        if strategy == "fixed_z":
            poisson_values = np.asarray([
                fixed_z_poisson_sensitivity(n, alpha_to_plot, single_target, response)
                for n in dense_n
            ])
            poisson_line, = ax.plot(
                dense_n, poisson_values, color=meta["color"],
                linestyle=":", linewidth=2.1, alpha=0.98, zorder=4,
            )
            single_legend_handles[strategy] = poisson_line
        continue

    fitted = adaptive_tail_power_law_fit(n_values, sensitivities)
    if fitted is not None:
        fit_n = np.geomspace(float(fitted["n_values"][0]), float(fitted["n_values"][-1]), 180)
        fit_values = np.exp(fitted["intercept"]) * fit_n ** fitted["slope"]
        fit_line, = ax.plot(
            fit_n, fit_values, color=meta["color"], linestyle="--",
            linewidth=2.0, alpha=0.95, zorder=3,
        )
        fit_key = f"{strategy}_fit"
        single_legend_handles[fit_key] = fit_line
        adaptive_fit_labels[fit_key] = rf"{meta['label']} fit ($s={fitted['slope']:.2f}$)"

ax.set_xscale("log", base=2)
ax.set_yscale("log")
ax.set_xlim(single_n_min, single_n_max)
ax.set_ylim(1.0e-2, 1.0)
tick_min = int(math.ceil(math.log2(single_n_min)))
tick_max = int(math.floor(math.log2(single_n_max)))
tick_powers = np.arange(tick_min, tick_max + 1, 2, dtype=int)
if tick_powers[-1] != tick_max:
    tick_powers = np.append(tick_powers, tick_max)
x_ticks = 2.0 ** tick_powers
ax.set_xticks(x_ticks)
ax.set_xticklabels([rf"$2^{{{power}}}$" for power in tick_powers])
ax.set_yticks([1.0e-2, 1.0e-1, 1.0])
ax.set_xlabel(r"Number of shots $n$", fontsize=17)
ax.set_ylabel(r"Sensitivity on $\Phi_R$", fontsize=17)
ax.grid(False, which="both")
ax.tick_params(axis="both", which="major", labelsize=15, direction="in", length=5.5, top=True, right=True)
ax.tick_params(axis="both", which="minor", direction="in", length=3.0, top=True, right=True)
single_legend_order = []
for strategy in single_adaptive_strategies:
    single_legend_order.append(strategy)
    if f"{strategy}_fit" in adaptive_fit_labels:
        single_legend_order.append(f"{strategy}_fit")
single_legend_order.extend(["two_axis", "fixed_z", ORACLE_KEY])
single_legend_order = [
    strategy for strategy in single_legend_order if strategy in single_legend_handles
]
legend_labels = [
    adaptive_fit_labels[strategy] if strategy in adaptive_fit_labels else all_styles[strategy]["label"]
    for strategy in single_legend_order
]
ax.legend(
    [single_legend_handles[strategy] for strategy in single_legend_order],
    legend_labels, frameon=False, fontsize=13.2, loc="lower left",
)
fig.tight_layout()

target_token = format(single_target, ".3g").replace(".", "p")
single_pdf_path = FIGURE_DIR / f"{OUTPUT_STEM}_sensitivity_PH1_{target_token}.pdf"
fig.savefig(single_pdf_path, bbox_inches="tight")
print(f"saved {single_pdf_path}")
plt.show()


In [ ]:
summary = {
    "raw_cache_path": str(RAW_CACHE_PATH),
    "raw_setup_hash": RAW_SETUP_HASH,
    "output_tag": OUTPUT_TAG,
    "raw_setup": raw_setup,
    "z_stars": Z_STARS.tolist(),
    "alphas": ALPHAS.tolist(),
    "target_powers": list(TARGET_POWERS),
    "discovery_n_shots": DISCOVERY_N_SHOTS,
}
summary_path = FIGURE_DIR / f"{OUTPUT_STEM}_summary.json"
power_csv_path = FIGURE_DIR / f"{OUTPUT_STEM}_power.csv"
sensitivity_csv_path = FIGURE_DIR / f"{OUTPUT_STEM}_sensitivity.csv"
fit_csv_path = FIGURE_DIR / f"{OUTPUT_STEM}_adaptive_fit.csv"
analytic_csv_path = FIGURE_DIR / f"{OUTPUT_STEM}_analytic_curves.csv"
summary_path.write_text(json.dumps(summary, indent=2))
combined_power_df.to_csv(power_csv_path, index=False)
sensitivity_df.to_csv(sensitivity_csv_path, index=False)
sensitivity_fit_df.to_csv(fit_csv_path, index=False)
sensitivity_analytic_df.to_csv(analytic_csv_path, index=False)
print(f"saved {summary_path}")
print(f"saved {power_csv_path}")
print(f"saved {sensitivity_csv_path}")
print(f"saved {fit_csv_path}")
print(f"saved {analytic_csv_path}")


In [ ]:
def first_log_bracketed_crossing(
    difference,
    n_min: float = 2.0**6,
    n_max: float = 2.0**22,
    grid_count: int = 1800,
) -> float:
    log_grid = np.linspace(math.log(n_min), math.log(n_max), int(grid_count))
    values = np.asarray([difference(math.exp(log_n)) for log_n in log_grid])
    valid = np.isfinite(values[:-1]) & np.isfinite(values[1:])
    brackets = np.flatnonzero(valid & (values[:-1] * values[1:] <= 0.0))
    if brackets.size == 0:
        return float("nan")
    index = int(brackets[0])
    root_log_n = brentq(
        lambda log_n: difference(math.exp(log_n)),
        float(log_grid[index]),
        float(log_grid[index + 1]),
    )
    return float(math.exp(root_log_n))


crossing_rows = []
adaptive_fits = sensitivity_fit_df[
    sensitivity_fit_df["strategy"].isin(ADAPTIVE_STRATEGIES)
    & np.isclose(sensitivity_fit_df["z_star"], z_to_plot)
].sort_values(["strategy", "target_power"])

for _, fit_row in adaptive_fits.iterrows():
    strategy = str(fit_row["strategy"])
    target = float(fit_row["target_power"])
    slope = float(fit_row["fitted_slope"])
    prefactor = float(math.exp(fit_row["intercept"]))
    fixed_z_at_one, _ = analytic_sensitivity(
        "fixed_z", np.asarray([1.0]), alpha_to_plot, target, response
    )
    fixed_z_prefactor = float(fixed_z_at_one[0])
    exponent_difference = slope + 0.25
    asymptotic_crossing = (
        float((fixed_z_prefactor / prefactor) ** (1.0 / exponent_difference))
        if abs(exponent_difference) > 1.0e-12
        else float("nan")
    )

    def poisson_difference(n_shots: float) -> float:
        adaptive_sensitivity = prefactor * float(n_shots) ** slope
        fixed_z_sensitivity = fixed_z_poisson_sensitivity(
            float(n_shots), alpha_to_plot, target, response
        )
        return adaptive_sensitivity - fixed_z_sensitivity

    poisson_crossing = first_log_bracketed_crossing(poisson_difference)
    fit_n_min = float(fit_row["n_fit_min"])
    fit_n_max = float(fit_row["n_fit_max"])
    crossing_rows.append({
        "strategy": strategy,
        "target_power": target,
        "adaptive_prefactor": prefactor,
        "adaptive_slope": slope,
        "fit_n_min": fit_n_min,
        "fit_n_max": fit_n_max,
        "n_z_adapt_asymptotic_z": asymptotic_crossing,
        "n_z_adapt_poisson_z": poisson_crossing,
        "poisson_crossing_in_fit_range": bool(
            np.isfinite(poisson_crossing)
            and fit_n_min <= poisson_crossing <= fit_n_max
        ),
    })

adaptive_crossing_df = pd.DataFrame(crossing_rows)
crossing_csv_path = FIGURE_DIR / f"{OUTPUT_STEM}_adaptive_fixed_z_crossing.csv"
adaptive_crossing_df.to_csv(crossing_csv_path, index=False)
display(adaptive_crossing_df.style.format({
    "target_power": "{:.2f}",
    "adaptive_prefactor": "{:.4f}",
    "adaptive_slope": "{:.4f}",
    "fit_n_min": "{:.0f}",
    "fit_n_max": "{:.0f}",
    "n_z_adapt_asymptotic_z": "{:.3g}",
    "n_z_adapt_poisson_z": "{:.3g}",
}))
print("Table IV uses n_z_adapt_poisson_z (the finite-n fixed-z curve shown in the final plot).")
for row in crossing_rows:
    print(
        f"{row['strategy']}, P_H1={row['target_power']:.2f}: "
        f"n_z:adapt={row['n_z_adapt_poisson_z']:.3g} "
        f"(asymptotic-z reference {row['n_z_adapt_asymptotic_z']:.3g})"
    )
print(f"saved {crossing_csv_path}")


In [ ]:
def h1_polarization_efficiency_samples(
    design: np.ndarray,
    true_phases: np.ndarray,
    checkpoints: np.ndarray,
) -> np.ndarray:
    # Arrays are shaped (amplitude, experiment, checkpoint, 3).
    design = np.asarray(design, dtype=float)
    true_phases = np.asarray(true_phases, dtype=float)
    checkpoints = np.asarray(checkpoints, dtype=float)
    cosine = np.cos(true_phases)[:, :, None]
    sine = np.sin(true_phases)[:, :, None]
    projected = (
        design[..., 0] * cosine**2
        + 2.0 * design[..., 1] * cosine * sine
        + design[..., 2] * sine**2
    )
    return 2.0 * projected / checkpoints[None, None, :]


def h0_phase_averaged_polarization_efficiency(
    design: np.ndarray,
    checkpoints: np.ndarray,
) -> np.ndarray:
    # Average f_pol over an a-priori uniform signal direction.
    design = np.asarray(design, dtype=float)
    checkpoints = np.asarray(checkpoints, dtype=float)
    return (design[..., 0] + design[..., 2]) / checkpoints[None, :]


polarization_grid_rows = []
fpol_h1_by_strategy = {}
for strategy in RAW_STRATEGIES:
    fpol_h1 = h1_polarization_efficiency_samples(
        raw["polarization_design_h1"][strategy],
        raw["true_drive_phases_h1"][strategy],
        CHECKPOINTS,
    )
    fpol_h1_by_strategy[strategy] = fpol_h1
    fpol_h0 = h0_phase_averaged_polarization_efficiency(
        raw["polarization_design_h0"][strategy],
        CHECKPOINTS,
    )
    for phase_index, phase_amplitude in enumerate(PHASE_GRID):
        for checkpoint_index, n_shots in enumerate(CHECKPOINTS):
            samples = fpol_h1[phase_index, :, checkpoint_index]
            polarization_grid_rows.append({
                "strategy": strategy,
                "phase_amplitude": float(phase_amplitude),
                "n_shots": int(n_shots),
                "f_pol_h1_mean": float(np.mean(samples)),
                "f_pol_h1_se": float(np.std(samples, ddof=1) / math.sqrt(samples.size)),
                "f_pol_h0_phase_mean": float(np.mean(fpol_h0[:, checkpoint_index])),
                "f_pol_h0_phase_mean_se": float(
                    np.std(fpol_h0[:, checkpoint_index], ddof=1)
                    / math.sqrt(fpol_h0.shape[0])
                ),
            })

polarization_grid_df = pd.DataFrame(polarization_grid_rows)

# Use only amplitudes explicitly present in the raw cache: no sensitivity
# crossing or interpolation is needed for this diagnostic.
selected_phase_indices = []
for requested_phase in FPOL_PHASES:
    matches = np.flatnonzero(np.isclose(PHASE_GRID, requested_phase, rtol=0.0, atol=1.0e-12))
    if matches.size != 1:
        raise ValueError(
            f"Requested f_pol phase {requested_phase:g} is not an exact cached value; "
            f"available values are {PHASE_GRID.tolist()}"
        )
    selected_phase_indices.append(int(matches[0]))
SELECTED_FPOL_PHASES = PHASE_GRID[np.asarray(selected_phase_indices, dtype=int)]

polarization_df = polarization_grid_df[
    polarization_grid_df["strategy"].isin(ADAPTIVE_STRATEGIES)
    & polarization_grid_df["phase_amplitude"].isin(SELECTED_FPOL_PHASES)
].copy()
polarization_df = polarization_df[[
    "strategy", "phase_amplitude", "n_shots",
    "f_pol_h1_mean", "f_pol_h1_se",
]].rename(columns={
    "f_pol_h1_mean": "f_pol",
    "f_pol_h1_se": "f_pol_se",
})
polarization_df["source"] = "direct H1 design at fixed injected amplitude"
polarization_grid_csv_path = FIGURE_DIR / f"{OUTPUT_STEM}_polarization_efficiency_grid.csv"
polarization_csv_path = FIGURE_DIR / f"{OUTPUT_STEM}_polarization_efficiency.csv"
polarization_grid_df.to_csv(polarization_grid_csv_path, index=False)
polarization_df.to_csv(polarization_csv_path, index=False)

fig, ax = plt.subplots(figsize=(8.6, 6.2))
phase_colors = plt.cm.viridis(np.linspace(0.16, 0.84, len(SELECTED_FPOL_PHASES)))
for strategy in ADAPTIVE_STRATEGIES:
    meta = STYLE[strategy]
    for color, phase_amplitude in zip(phase_colors[::-1], SELECTED_FPOL_PHASES[::-1]):
        rows = polarization_df[
            (polarization_df["strategy"] == strategy)
            & np.isclose(polarization_df["phase_amplitude"], float(phase_amplitude))
        ].sort_values("n_shots")
        ax.errorbar(
            rows["n_shots"], rows["f_pol"], yerr=rows["f_pol_se"],
            marker=meta["marker"], markerfacecolor="white", markeredgewidth=1.4,
            markersize=5.5, linestyle="-", linewidth=1.5, capsize=2.5,
            capthick=1.1, elinewidth=1.1, color=color,
            alpha=1.0 if strategy == ADAPTIVE_STRATEGIES[0] else 0.65,
            label=rf"{meta['label']}, $\Phi_R={float(phase_amplitude):g}$",
        )
ax.axhline(1.0, color="0.35", linestyle=":", linewidth=1.4)
ax.axhline(2.0, color="black", linestyle="--", linewidth=1.4)
ax.set_xscale("log", base=2)
ax.set_xlim(float(CHECKPOINTS.min()), float(CHECKPOINTS.max()))
ax.set_ylim(0.0, 2.1)
ax.set_xlabel(r"Number of shots $n$", fontsize=21)
ax.set_ylabel(r"Polarization efficiency $f_{\rm pol}$", fontsize=21)
ax.tick_params(axis="both", which="major", labelsize=18, direction="in", length=7, top=True, right=True)
ax.tick_params(axis="both", which="minor", direction="in", length=4, top=True, right=True)
ax.grid(False)
ax.legend(frameon=False, fontsize=15, ncol=1, loc="upper left", bbox_to_anchor=(0.01, 0.93))
fig.tight_layout()
polarization_pdf_path = FIGURE_DIR / f"{OUTPUT_STEM}_polarization_efficiency.pdf"
fig.savefig(polarization_pdf_path, bbox_inches="tight")
print(f"saved {polarization_grid_csv_path}")
print(f"saved {polarization_csv_path}")
print(f"saved {polarization_pdf_path}")
display(polarization_df)
plt.show()
